# Объединение чеков и магазинов.
Ноутбук делает три вещи:

1. Загружает файлы `store_checkout_queues` и `store_stores` из локальной папки.
2. Объединяет данные по `store_uuid`, выбирает магазины `98451680` и `12864064`.
3. Выгружает результат в Excel.



# 1. Импорт библиотек и пути к файлам
Исходные файлы лежат в папке:

C:\Users\Users\Desktop\Новая папка

Названия исходных файлов не меняем.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_DIR = Path(r"C:\Users\Users\Desktop\Новая папка")

SCQ_FILE = "store_checkout_queues.csv"
SS_FILE = "store_stores.csv"
SOLUTION_FILE = "store_queue_solution.xlsx"  # контрольный файл, в расчётах не используется

scq_path = DATA_DIR / SCQ_FILE
ss_path = DATA_DIR / SS_FILE
solution_path = DATA_DIR / SOLUTION_FILE

print("Файл чеков:", scq_path)
print("Файл магазинов:", ss_path)

Файл чеков: C:\Users\Users\Desktop\Новая папка\store_checkout_queues.csv
Файл магазинов: C:\Users\Users\Desktop\Новая папка\store_stores.csv


# 2. Чтение CSV-файлов
Функция ниже читает CSV устойчиво: пробует несколько популярных кодировок и автоматически определяет разделитель.

In [3]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    """
    Безопасное чтение CSV.

    sep=None + engine='python' позволяют pandas самому определить разделитель.
    Кодировки:
    - utf-8-sig: частый вариант CSV из Excel;
    - utf-8: стандартная кодировка;
    - cp1251: частая кодировка русскоязычных Windows-файлов.
    """
    encodings = ["utf-8-sig", "utf-8", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(
                path,
                sep=None,
                engine="python",
                encoding=encoding,
            )
        except Exception as error:
            last_error = error

    raise RuntimeError(
        f"Не удалось прочитать файл: {path}\n"
        f"Последняя ошибка: {last_error}"
    )

In [4]:
scq_raw = read_csv_safely(scq_path)
ss_raw = read_csv_safely(ss_path)

print("store_checkout_queues:", scq_raw.shape)
print("store_stores:", ss_raw.shape)

display(scq_raw.head())
display(ss_raw.head())

store_checkout_queues: (1064703, 8)
store_stores: (68, 2)


,checks_number,store_uuid,employees_id,quantity,selling_price,checkout_id1,start_operation_dt,end_operation_dt
0,787514,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,02.12.2024 10:13:23,02.12.2024 10:13:31
1,19753765,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,22.12.2024 14:12:31,22.12.2024 14:12:46
2,24941583,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,27.11.2024 12:57:10,27.11.2024 12:57:40
3,10309468,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,12.12.2024 13:22:33,12.12.2024 13:23:00
4,5654127,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,07.12.2024 12:30:44,07.12.2024 12:31:05


,store_id,store_uuid
0,10722670,712f0db6-2493-11e9-95e8-0050569efd74
1,33697056,pzah4kje-acfz-zzgf-q3g0-yyttd3dk08
2,12073765,823a1fc7-35a4-22fa-96f9-1161678afd75
3,79878639,qabi5lkf-bdga-aahg-r4h1-zzuue4el09
4,24985584,934b2ed8-46b5-33eb-a7ea-2272789bfe76


# 3. Нормализация названий столбцов
На всякий случай приводим названия колонок к единому стилю:

убираем пробелы;
переводим в нижний регистр;
заменяем пробелы на _;
убираем возможные префиксы scq и ss.

In [5]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Приводит названия колонок к единому виду.
    Это снижает риск ошибки из-за регистра, пробелов или префиксов.
    """
    result = df.copy()
    normalized_columns = []

    for column in result.columns:
        column_name = str(column).strip().lower().replace(" ", "_")

        for prefix in ("scq.", "ss.", "scq_", "ss_"):
            if column_name.startswith(prefix):
                column_name = column_name[len(prefix):]

        normalized_columns.append(column_name)

    result.columns = normalized_columns
    return result


scq = normalize_columns(scq_raw)
ss = normalize_columns(ss_raw)

print("Колонки store_checkout_queues:")
print(list(scq.columns))

print("\nКолонки store_stores:")
print(list(ss.columns))

Колонки store_checkout_queues:
['checks_number', 'store_uuid', 'employees_id', 'quantity', 'selling_price', 'checkout_id1', 'start_operation_dt', 'end_operation_dt']

Колонки store_stores:
['store_id', 'store_uuid']


# 4.Подготовка фактических названий полей
По данным видно, что в store_checkout_queues поле кассы называется checkout_id1.

Для результата по условию нужно название checkout_id, поэтому переименовываем колонку.

Также store_id находится не в таблице чеков, а в таблице магазинов.
Поэтому объединение делаем через store_uuid.

In [6]:
# В таблице чеков поле кассы называется checkout_id1.
# Для финальной таблицы по условию переименовываем его в checkout_id.
if "checkout_id1" in scq.columns and "checkout_id" not in scq.columns:
    scq = scq.rename(columns={"checkout_id1": "checkout_id"})

print("Колонки store_checkout_queues после переименования:")
print(list(scq.columns))

Колонки store_checkout_queues после переименования:
['checks_number', 'store_uuid', 'employees_id', 'quantity', 'selling_price', 'checkout_id', 'start_operation_dt', 'end_operation_dt']


In [7]:
output_columns = [
    "checks_number",
    "store_id",
    "employees_id",
    "quantity",
    "selling_price",
    "checkout_id",
    "start_operation_dt",
    "end_operation_dt",
]

required_scq_columns = [
    "checks_number",
    "store_uuid",
    "employees_id",
    "quantity",
    "selling_price",
    "checkout_id",
    "start_operation_dt",
    "end_operation_dt",
]

required_ss_columns = [
    "store_id",
    "store_uuid",
]

missing_scq_columns = [column for column in required_scq_columns if column not in scq.columns]
missing_ss_columns = [column for column in required_ss_columns if column not in ss.columns]

if missing_scq_columns:
    raise KeyError(
        "В store_checkout_queues не хватает столбцов: "
        + ", ".join(missing_scq_columns)
    )

if missing_ss_columns:
    raise KeyError(
        "В store_stores не хватает столбцов: "
        + ", ".join(missing_ss_columns)
    )

print("Все обязательные столбцы найдены.")

Все обязательные столбцы найдены.


# 6.Подготовка типов данных
store_id нужен для фильтрации по магазинам.
store_uuid оставляем строкой, чтобы объединение было стабильным.

Даты переводим в datetime, числовые поля — в числовой формат.

In [8]:
# Ключ для объединения
scq["store_uuid"] = scq["store_uuid"].astype(str)
ss["store_uuid"] = ss["store_uuid"].astype(str)

# Идентификатор магазина для фильтрации
ss["store_id"] = pd.to_numeric(ss["store_id"], errors="raise").astype("int64")

# Числовые поля
numeric_columns = ["quantity", "selling_price"]

for column in numeric_columns:
    scq[column] = pd.to_numeric(scq[column], errors="coerce")

# Даты начала и окончания операции.
# В исходном CSV формат фиксированный: день.месяц.год часы:минуты:секунды.
# Явный format исключает ошибочное распознавание дня как месяца.
datetime_columns = ["start_operation_dt", "end_operation_dt"]

for column in datetime_columns:
    missing_before = scq[column].isna().sum()

    scq[column] = pd.to_datetime(
        scq[column],
        format="%d.%m.%Y %H:%M:%S",
        errors="raise",
    )

    missing_after = scq[column].isna().sum()
    assert missing_after == missing_before, (
        f"При преобразовании {column} появились новые пропуски: "
        f"{missing_after - missing_before}"
    )

print("Типы данных подготовлены.")

Типы данных подготовлены.


# 7. Объединение таблиц
Объединяем:

store_checkout_queues ← store_stores

Ключ объединения:

store_uuid

Из справочника магазинов подтягиваем store_id.

In [9]:
stores_for_merge = (
    ss[["store_uuid", "store_id"]]
    .drop_duplicates(subset=["store_uuid"])
    .copy()
)

checks_with_stores = scq.merge(
    stores_for_merge,
    on="store_uuid",
    how="inner",
    validate="many_to_one",
)

print("Строк в store_checkout_queues до объединения:", len(scq))
print("Строк после объединения с store_stores:", len(checks_with_stores))

display(checks_with_stores.head())

Строк в store_checkout_queues до объединения: 1064703
Строк после объединения с store_stores: 1064703


,checks_number,store_uuid,employees_id,quantity,selling_price,checkout_id,start_operation_dt,end_operation_dt,store_id
0,787514,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,2024-12-02 10:13:23,2024-12-02 10:13:31,19045434
1,19753765,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,2024-12-22 14:12:31,2024-12-22 14:12:46,19045434
2,24941583,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,2024-11-27 12:57:10,2024-11-27 12:57:40,19045434
3,10309468,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,2024-12-12 13:22:33,2024-12-12 13:23:00,19045434
4,5654127,tdel8oni-egjd-ddij-u7k4-ccxxh7ho12,121140,4,271,6,2024-12-07 12:30:44,2024-12-07 12:31:05,19045434


# 8. Фильтрация по магазинам из условия
Оставляем только магазины:

98451680
12864064
После этого выбираем только нужные поля в нужном порядке.

In [10]:
target_store_ids = [98451680, 12864064]

queue_df = (
    checks_with_stores
    .loc[checks_with_stores["store_id"].isin(target_store_ids), output_columns]
    .copy()
    .sort_values(
        by=["store_id", "start_operation_dt", "checks_number"],
        ignore_index=True,
    )
)

print("Итоговый размер датафрейма:", queue_df.shape)
display(queue_df.head(20))

Итоговый размер датафрейма: (50581, 8)


,checks_number,store_id,employees_id,quantity,selling_price,checkout_id,start_operation_dt,end_operation_dt
0,18928481,12864064,599,4,34,18,2024-11-21 11:41:23,2024-11-21 11:42:02
1,18928491,12864064,599,4,148,16,2024-11-21 11:48:26,2024-11-21 11:49:18
2,18928699,12864064,71646,4,159,8,2024-11-21 14:05:54,2024-11-21 14:06:13
3,18928735,12864064,71646,4,479,8,2024-11-21 14:39:31,2024-11-21 14:40:06
4,18928744,12864064,71646,5,1194,8,2024-11-21 14:44:05,2024-11-21 14:46:54
5,18928779,12864064,178152,4,504,7,2024-11-21 15:15:19,2024-11-21 15:16:16
6,18928873,12864064,178152,3,49,7,2024-11-21 16:33:02,2024-11-21 16:34:08
7,18928926,12864064,599,4,152,18,2024-11-21 17:10:02,2024-11-21 17:10:32
8,18929090,12864064,178152,4,237,7,2024-11-21 18:37:43,2024-11-21 18:38:32
9,18929138,12864064,178152,4,244,7,2024-11-21 19:06:07,2024-11-21 19:06:26


# 9. Контроль результата
Проверяем:

порядок колонок;
отсутствие лишних магазинов;
количество строк по каждому магазину.

In [11]:
assert list(queue_df.columns) == output_columns, "Порядок или состав столбцов не соответствует заданию."

result_store_ids = set(queue_df["store_id"].unique())
expected_store_ids = set(target_store_ids)

assert result_store_ids.issubset(expected_store_ids), "В результате есть магазины не из условия."

print("Магазины в результате:", sorted(result_store_ids))

print("\nКоличество строк по магазинам:")
display(
    queue_df["store_id"]
    .value_counts()
    .rename_axis("store_id")
    .reset_index(name="rows_count")
    .sort_values("store_id", ignore_index=True)
)

print("\nПропуски в итоговой таблице:")
display(queue_df.isna().sum().to_frame("missing_values"))

Магазины в результате: [12864064, 98451680]

Количество строк по магазинам:


,store_id,rows_count
0,12864064,26387
1,98451680,24194



Пропуски в итоговой таблице:


,missing_values
checks_number,0
store_id,0
employees_id,0
quantity,0
selling_price,0
checkout_id,0
start_operation_dt,0
end_operation_dt,0


# 10. Сохранение результата
Исходные файлы не изменяются.

Результат сохраняется отдельными файлами:

store_queue_prepared.csv
store_queue_prepared.xlsx
Главный датафрейм для дальнейшей работы: queue_df.

In [12]:
output_csv_path = DATA_DIR / "store_queue_prepared.csv"
output_excel_path = DATA_DIR / "store_queue_prepared.xlsx"

queue_df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
print("CSV сохранён:", output_csv_path)

try:
    queue_df.to_excel(output_excel_path, index=False)
    print("Excel сохранён:", output_excel_path)
except ImportError as error:
    print("Excel-файл не сохранён: в окружении не хватает библиотеки для записи .xlsx.")
    print("Установи openpyxl командой: pip install openpyxl")
    print("Текст ошибки:", error)

CSV сохранён: C:\Users\Arthursus\Desktop\тест ещё один\store_queue_prepared.csv
Excel сохранён: C:\Users\Arthursus\Desktop\тест ещё один\store_queue_prepared.xlsx


# 11. Финальный результат
В результате получен датафрейм queue_df:

чеки объединены с магазинами;
store_id подтянут из таблицы store_stores;
оставлены только магазины 98451680 и 12864064;
колонка checkout_id1 приведена к названию checkout_id;
порядок столбцов соответствует условию.